# LeetCode 688 — Knight Probability in Chessboard bằng Python

Link đề: https://leetcode.com/problems/knight-probability-in-chessboard/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def knightProbability(self, n: int, k: int, row: int, column: int) -> float:
        ...
```

Ghi chú: kết quả là **số thực**, LeetCode chấp nhận sai số tuyệt đối khoảng `1e-5`; trong notebook ta so sánh có sai số và đối chiếu với kết quả **chính xác** dưới dạng phân số (`fractions.Fraction`). Ví dụ 3, 4, 5 trong mục 2 do notebook tự đặt, đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: quân mã đứng ở ô `(row, column)` trên bàn `n x n`, mỗi lượt chọn **đều ngẫu nhiên** một trong 8 nước đi (kể cả nước đi ra khỏi bàn), đi đúng `k` nước hoặc dừng khi đã rơi ra ngoài; tính **xác suất mã còn ở trên bàn** sau khi dừng.
2. Nắm các công cụ Python: danh sách 8 hướng đi của quân mã, kiểm tra biên `0 <= x < n`, mảng 2 chiều/3 chiều, `functools.lru_cache`, mảng "cuốn chiếu" (rolling array), `fractions.Fraction` để tính chính xác, và so sánh số thực bằng sai số.
3. Đi từ **liệt kê mọi đường đi (8^k)** → **đệ quy có nhớ (top-down)** → **quy hoạch động từng bước "kéo" (pull) với mảng cuốn chiếu** → **quy hoạch động "đẩy" (push) phân phối xác suất từ ô xuất phát** → **tận dụng đối xứng của bàn cờ (chỉ tính 1/8 số ô)**.
4. Hiểu mấu chốt: xác suất sống sót sau `h` nước chỉ phụ thuộc **ô hiện tại** và **số nước còn lại**, nên `f(h, i, j) = (1/8) · Σ f(h-1, ô kề)` với `f(0, ô trong bàn) = 1` và `f(·, ô ngoài bàn) = 0`. Có `k · n²` trạng thái thay vì `8^k` đường đi.
5. Tránh các lỗi kinh điển: chia cho "số nước hợp lệ" thay vì chia cho `8`, quên `k = 0`, tạo bảng 2 chiều bằng `[[0] * n] * n` (các hàng dùng chung), đệ quy không nhớ bị quá thời gian, và so sánh số thực bằng `==`.

## 2. Đề bài & ví dụ

Trên bàn cờ `n x n`, một quân mã bắt đầu ở ô `(row, column)` và thực hiện đúng `k` nước đi. Hàng và cột đánh số từ `0` (ô trên-trái là `(0, 0)`, ô dưới-phải là `(n - 1, n - 1)`).

Quân mã có **8 nước đi** có thể: đi **2 ô** theo một hướng chính (lên/xuống/trái/phải) rồi **1 ô** theo hướng vuông góc. Mỗi lượt, mã chọn **đều ngẫu nhiên** một trong 8 nước (xác suất `1/8` mỗi nước), **kể cả khi nước đó đưa mã ra ngoài bàn**.

Mã tiếp tục đi tới khi đã đi đủ `k` nước **hoặc** đã ra khỏi bàn. Trả về **xác suất mã vẫn còn trên bàn** khi dừng.

**Ví dụ 1**

- Input: `n = 3`, `k = 2`, `row = 0`, `column = 0`
- Output: `0.06250`
- Giải thích: từ `(0, 0)` chỉ có 2 nước giữ mã trong bàn (tới `(1, 2)` và `(2, 1)`), và từ mỗi ô đó lại có 2 nước ở lại trong bàn. Xác suất `(2/8) · (2/8) = 0.0625`.

**Ví dụ 2**

- Input: `n = 1`, `k = 0`, `row = 0`, `column = 0`
- Output: `1.00000` (không đi nước nào nên mã chắc chắn còn trên bàn).

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `n = 1`, `k = 1`, `row = 0`, `column = 0`
- Output: `0.0` (bàn chỉ có 1 ô, mọi nước đi đều ra ngoài).

**Ví dụ 4 (tự đặt)**

- Input: `n = 3`, `k = 1`, `row = 1`, `column = 1`
- Output: `0.0` (mã ở tâm bàn 3x3 mọi nước đều ra ngoài).

**Ví dụ 5 (tự đặt)**

- Input: `n = 4`, `k = 1`, `row = 1`, `column = 1`
- Output: `0.5` (4 trong 8 nước vẫn nằm trong bàn).

**Ràng buộc:**

- `1 <= n <= 25`
- `0 <= k <= 100`
- `0 <= row, column <= n - 1`

Chú ý: đáp án là số thực; kết quả đúng nằm trong khoảng `[0, 1]`.

In [1]:
from typing import List
from functools import lru_cache
from fractions import Fraction
from itertools import product
import random
import time

MOVES = [(-2, -1), (-2, 1), (-1, -2), (-1, 2), (1, -2), (1, 2), (2, -1), (2, 1)]

def close(a, b, tol=1e-9):
    """So sánh hai số thực có sai số tuyệt đối tol (LeetCode cho phép khoảng 1e-5)."""
    return abs(a - b) <= tol

examples = [
    (3, 2, 0, 0, 0.0625),
    (1, 0, 0, 0, 1.0),
    (1, 1, 0, 0, 0.0),
    (3, 1, 1, 1, 0.0),
    (4, 1, 1, 1, 0.5),
]
for n, k, r, c, expected in examples:
    print(f"n={n}, k={k}, (row, col)=({r},{c}) -> kỳ vọng {expected}")

n=3, k=2, (row, col)=(0,0) -> kỳ vọng 0.0625
n=1, k=0, (row, col)=(0,0) -> kỳ vọng 1.0
n=1, k=1, (row, col)=(0,0) -> kỳ vọng 0.0
n=3, k=1, (row, col)=(1,1) -> kỳ vọng 0.0
n=4, k=1, (row, col)=(1,1) -> kỳ vọng 0.5


## 3. Kiến thức nền

### 3.1. Tám nước đi của quân mã và kiểm tra biên

Nước đi của mã là `(dr, dc)` với `{|dr|, |dc|} = {1, 2}`: có đúng 8 cặp. Từ ô `(i, j)` ta tới `(i + dr, j + dc)`. Ô đó nằm **trên bàn** khi `0 <= x < n` và `0 <= y < n` (Python cho viết gọn `0 <= x < n`).

In [2]:
print(len(MOVES), MOVES)
n = 3
i, j = 0, 0
inside = [(i + dr, j + dc) for dr, dc in MOVES if 0 <= i + dr < n and 0 <= j + dc < n]
print("từ (0,0) trên bàn 3x3 còn các ô:", inside)
print("số nước hợp lệ:", len(inside), "trên tổng 8 -> xác suất ở lại sau 1 nước =", len(inside) / 8)

8 [(-2, -1), (-2, 1), (-1, -2), (-1, 2), (1, -2), (1, 2), (2, -1), (2, 1)]
từ (0,0) trên bàn 3x3 còn các ô: [(1, 2), (2, 1)]
số nước hợp lệ: 2 trên tổng 8 -> xác suất ở lại sau 1 nước = 0.25


### 3.2. Xác suất: nhân theo chuỗi, cộng theo nhánh

- **Quy tắc nhân:** xác suất của một chuỗi lựa chọn độc lập là tích xác suất từng bước. Một đường đi cụ thể gồm `k` nước có xác suất `(1/8)^k`.
- **Quy tắc cộng:** các đường đi khác nhau là các biến cố **loại trừ nhau**, nên xác suất "mã còn trên bàn" là tổng xác suất mọi đường đi còn trên bàn: `số đường đi hợp lệ / 8^k`.

Lưu ý "đường đi hợp lệ" ở đây gồm `k` nước mà **mọi ô đi qua đều trong bàn** (nếu một nước ra ngoài, mã dừng và đã mất).

In [3]:
k = 2
print("xác suất một đường đi cụ thể:", (1 / 8) ** k, "=", 1 / 8 ** k)
print("tổng số đường đi (kể cả ra ngoài):", 8 ** k)
print("ví dụ 1: 2 nước ban đầu * 2 nước tiếp = 4 đường hợp lệ ->", 4 / 8 ** k)

xác suất một đường đi cụ thể: 0.015625 = 0.015625
tổng số đường đi (kể cả ra ngoài): 64
ví dụ 1: 2 nước ban đầu * 2 nước tiếp = 4 đường hợp lệ -> 0.0625


### 3.3. Bảng 2 chiều / 3 chiều và cái bẫy `[[0] * n] * n`

Bảng `n x n` đúng cách: `[[0] * n for _ in range(n)]`. Viết `[[0] * n] * n` tạo `n` tham chiếu tới **cùng một hàng**, nên sửa một ô sẽ sửa cả cột đó ở mọi hàng. Bảng 3 chiều cho `k + 1` tầng: `[[[0] * n for _ in range(n)] for _ in range(k + 1)]`.

In [4]:
bad = [[0] * 3] * 3
bad[0][0] = 7
print("sai :", bad)
good = [[0] * 3 for _ in range(3)]
good[0][0] = 7
print("đúng:", good)

sai : [[7, 0, 0], [7, 0, 0], [7, 0, 0]]
đúng: [[7, 0, 0], [0, 0, 0], [0, 0, 0]]


### 3.4. `lru_cache` (nhớ kết quả đệ quy) và mảng cuốn chiếu

`functools.lru_cache(maxsize=None)` đặt trên một hàm đệ quy sẽ **nhớ** kết quả theo tham số: lần sau gọi lại cùng tham số trả ngay kết quả. Tham số phải **băm được** (số, tuple...). Mảng **cuốn chiếu**: vì tầng `h` chỉ cần tầng `h - 1`, ta chỉ giữ hai bảng `n x n` thay vì `k + 1` bảng.

In [5]:
calls = 0

@lru_cache(maxsize=None)
def fib(m):
    global calls
    calls += 1
    return m if m < 2 else fib(m - 1) + fib(m - 2)

print(fib(30), "chỉ gọi thực sự", calls, "lần (không có nhớ sẽ là hàng triệu lần)")

832040 chỉ gọi thực sự 31 lần (không có nhớ sẽ là hàng triệu lần)


### 3.5. Số thực (float), sai số và `Fraction`

Số thực `float` lưu xấp xỉ nhị phân nên phép cộng/chia có sai số rất nhỏ (cỡ `1e-16` mỗi phép). Vì vậy **đừng so sánh `==`**; hãy so `abs(a - b) <= tol`. `fractions.Fraction(a, b)` biểu diễn **phân số chính xác**, dùng để kiểm tra: ta tính đáp án đúng dạng phân số rồi `float(...)`.

In [6]:
print(0.1 + 0.2 == 0.3, abs((0.1 + 0.2) - 0.3) <= 1e-9)
x = Fraction(1, 8) * Fraction(1, 8) * 4
print(x, float(x), x == Fraction(1, 16))
print(sum([1 / 8] * 8), sum([Fraction(1, 8)] * 8))

False True
1/16 0.0625 True
1.0 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Bàn `3 x 3`, mã ở `(0, 0)`, `k = 2`. Ta đánh số các ô:

```
(0,0) (0,1) (0,2)
(1,0) (1,1) (1,2)
(2,0) (2,1) (2,2)
```

**Sau 1 nước** (`h = 1`): với mỗi ô, đếm số nước đi vẫn trong bàn rồi chia 8.

| ô | các ô tới được trong bàn | xác suất ở lại sau 1 nước |
|---|---|---|
| góc `(0,0)` | `(1,2)`, `(2,1)` | `2/8` |
| cạnh `(0,1)` | `(2,0)`, `(2,2)` | `2/8` |
| tâm `(1,1)` | không có | `0` |

(Tất cả 8 ô quanh tâm đều có đúng 2 nước trong bàn nên `2/8 = 0.25`; tâm là `0`.)

**Sau 2 nước** từ `(0,0)`: mã đi tới `(1,2)` hoặc `(2,1)` (mỗi cái xác suất `1/8`), rồi từ đó còn xác suất `2/8` ở lại: `(1/8) · (2/8) + (1/8) · (2/8) = 4/64 = 0.0625`. Tức là: **xác suất sống sót `h` nước từ ô `u` = trung bình (chia 8) của xác suất sống sót `h-1` nước từ các ô kề `v` còn trên bàn**.

In [7]:
n = 3
def one_step(i, j):
    return sum(1 for dr, dc in MOVES if 0 <= i + dr < n and 0 <= j + dc < n) / 8

for i in range(n):
    print([one_step(i, j) for j in range(n)])
print("hai nước từ (0,0):", sum(one_step(0 + dr, 0 + dc) / 8 for dr, dc in MOVES if 0 <= dr < n and 0 <= dc < n))

[0.25, 0.25, 0.25]
[0.25, 0.0, 0.25]
[0.25, 0.25, 0.25]
hai nước từ (0,0): 0.0625


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi đường đi

### 5.1. Ý tưởng

Duyệt **mọi** chuỗi `k` nước (có `8^k` chuỗi). Đếm số chuỗi mà **mọi ô đi qua đều trong bàn**. Xác suất `= số chuỗi hợp lệ / 8^k`. Dùng số nguyên để đếm (chính xác) rồi chia một lần ở cuối bằng `Fraction`.

In [8]:
def knight_brute_count(n, k, r, c):
    """Số chuỗi k nước mà mã luôn ở trong bàn (đếm nguyên, đệ quy thuần)."""
    def go(i, j, steps):
        if not (0 <= i < n and 0 <= j < n):
            return 0
        if steps == k:
            return 1
        return sum(go(i + dr, j + dc, steps + 1) for dr, dc in MOVES)
    return go(r, c, 0)

def knight_brute(n, k, r, c):
    return knight_brute_count(n, k, r, c) / 8 ** k

for n, k, r, c, expected in examples:
    got = knight_brute(n, k, r, c)
    print(f"{n=} {k=} ({r},{c}) -> {got}  (đếm = {knight_brute_count(n, k, r, c)} / 8^{k})")
    assert close(got, expected)

n=3 k=2 (0,0) -> 0.0625  (đếm = 4 / 8^2)
n=1 k=0 (0,0) -> 1.0  (đếm = 1 / 8^0)
n=1 k=1 (0,0) -> 0.0  (đếm = 0 / 8^1)
n=3 k=1 (1,1) -> 0.0  (đếm = 0 / 8^1)
n=4 k=1 (1,1) -> 0.5  (đếm = 4 / 8^1)


### 5.2. Độ phức tạp và nút thắt

Số lời gọi đệ quy cỡ `8^k` (tối đa), tức Time `O(8^k)`, Space `O(k)` (độ sâu đệ quy). Với `k = 100` thì `8^100 ≈ 10^90` — không thể. Hãy xem số lời gọi tăng ra sao:

In [9]:
def count_calls(n, k, r, c):
    calls = 0
    def go(i, j, steps):
        nonlocal calls
        calls += 1
        if not (0 <= i < n and 0 <= j < n):
            return 0
        if steps == k:
            return 1
        return sum(go(i + dr, j + dc, steps + 1) for dr, dc in MOVES)
    go(r, c, 0)
    return calls

for k in range(1, 7):
    t0 = time.perf_counter()
    cnt = count_calls(8, k, 3, 3)
    print(f"k={k}: {cnt:8d} lời gọi, {1000 * (time.perf_counter() - t0):8.2f} ms")

k=1:        9 lời gọi,     0.01 ms
k=2:       73 lời gọi,     0.04 ms
k=3:      521 lời gọi,     0.13 ms
k=4:     3065 lời gọi,     0.65 ms
k=5:    18889 lời gọi,     4.50 ms
k=6:   112121 lời gọi,    26.41 ms


Nút thắt: ta **tính đi tính lại cùng một bài toán con** — "từ ô `(i, j)` còn `h` nước, xác suất sống sót?" — hàng nghìn lần qua các đường đi khác nhau. Ở bàn `n x n` chỉ có `k · n²` bài toán con khác nhau (tối đa `100 · 625 = 62 500`).

## 6. Ý tưởng thuật toán / bước đột phá

**Định nghĩa trạng thái.** `f(h, i, j)` = xác suất mã còn trên bàn sau khi đi `h` nước **bắt đầu từ ô `(i, j)`** (tức còn `h` nước phải đi).

**Điều kiện biên.**

- Nếu `(i, j)` **ngoài bàn**: `f = 0` (đã mất).
- Nếu `h = 0` và `(i, j)` trong bàn: `f = 1`.

**Chuyển trạng thái.** Với `h ≥ 1` và `(i, j)` trong bàn, mã chọn đều 8 nước:

`f(h, i, j) = (1/8) · Σ_{(dr,dc)} f(h - 1, i + dr, j + dc)`

(các ô ngoài bàn đóng góp `0`). **Đáp án** là `f(k, row, column)`.

**Vì sao đây là quy hoạch động:** `f(h, ·, ·)` chỉ phụ thuộc `f(h - 1, ·, ·)`, các nước đi sau độc lập với quá khứ (tính "không nhớ" của xích Markov). Có `(k + 1) · n²` trạng thái, mỗi trạng thái 8 phép cộng: `O(8 · k · n²)`.

**Ba biến thể tự nhiên:** (1) đệ quy có nhớ, (2) bảng từ `h = 0` lên `h = k` (kéo), (3) đi **xuôi** — bắt đầu từ ô xuất phát với xác suất 1, mỗi bước chia đều xác suất sang 8 ô kề (đẩy) và cuối cùng cộng xác suất còn trên bàn. Thêm một tối ưu: bàn cờ có **tính đối xứng** (8 phép: xoay/lật), nên `f` bằng nhau ở các ô đối xứng — chỉ cần tính khoảng 1/8 số ô.

## 7. Các cách tiếp theo

Tổng cộng có 5 cách: Cách 1 (liệt kê, mục 5) và bốn cách dưới đây. Mỗi cách: giải thích từng đoạn → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Đệ quy có nhớ (top-down)

Giải thích từng đoạn:

1. `f(h, i, j)` đúng như mục 6, dùng `@lru_cache(maxsize=None)` để mỗi bộ ba chỉ tính một lần.
2. Nếu `(i, j)` ngoài bàn trả `0.0`; nếu `h == 0` trả `1.0`.
3. Ngược lại `sum(f(h - 1, i + dr, j + dc) for dr, dc in MOVES) / 8`.
4. Đáp án `f(k, row, column)`. Độ sâu đệ quy `≤ k + 1 ≤ 101`, an toàn với giới hạn đệ quy mặc định.

In [10]:
def memo_demo(n, k, r, c):
    calls = {"count": 0}

    @lru_cache(maxsize=None)
    def f(h, i, j):
        calls["count"] += 1
        if not (0 <= i < n and 0 <= j < n):
            return 0.0
        if h == 0:
            return 1.0
        return sum(f(h - 1, i + dr, j + dc) for dr, dc in MOVES) / 8

    ans = f(k, r, c)
    return ans, calls["count"], f.cache_info().hits

print("n=3,k=2:", memo_demo(3, 2, 0, 0))
print("n=8,k=6:", memo_demo(8, 6, 3, 3), "(so với", count_calls(8, 6, 3, 3), "lời gọi ở cách 1)")

n=3,k=2: (0.0625, 23, 2)
n=8,k=6: (0.27028656005859375, 320, 737) (so với 112121 lời gọi ở cách 1)


In [11]:
class Sol2:
    """Cách 2: đệ quy có nhớ."""
    def knightProbability(self, n: int, k: int, row: int, column: int) -> float:
        @lru_cache(maxsize=None)
        def f(h, i, j):
            if not (0 <= i < n and 0 <= j < n):
                return 0.0
            if h == 0:
                return 1.0
            return sum(f(h - 1, i + dr, j + dc) for dr, dc in MOVES) / 8

        return f(k, row, column)

for n, k, r, c, expected in examples:
    assert close(Sol2().knightProbability(n, k, r, c), expected)
print("Sol2 OK")

Sol2 OK


**Độ phức tạp:** `(k + 1) · n²` trạng thái, mỗi trạng thái 8 phép cộng: Time `O(k · n²)`, Space `O(k · n²)` cho bộ nhớ đệm (cộng `O(k)` ngăn xếp).

### 7.2. Cách 3 — Quy hoạch động từng bước, kéo (pull) với mảng cuốn chiếu

Giải thích từng đoạn:

1. `cur[i][j]` = `f(h, i, j)`; khởi tạo `h = 0`: mọi ô trong bàn bằng `1.0`.
2. Lặp `h = 1..k`: `nxt[i][j] = (Σ cur[x][y] của các ô kề (x, y) trong bàn) / 8` (ô ngoài bàn góp 0, bỏ qua).
3. Gán `cur = nxt`. Sau `k` bước, trả `cur[row][column]`.

Chỉ giữ hai bảng `n x n` nên Space `O(n²)`.

In [12]:
def dp_pull_demo(n, k):
    cur = [[1.0] * n for _ in range(n)]
    print("h=0:", cur[0])
    for h in range(1, k + 1):
        nxt = [[0.0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                total = 0.0
                for dr, dc in MOVES:
                    x, y = i + dr, j + dc
                    if 0 <= x < n and 0 <= y < n:
                        total += cur[x][y]
                nxt[i][j] = total / 8
        cur = nxt
        print(f"h={h}:", [round(v, 5) for v in cur[0]], "... (hàng đầu)")
    return cur

table = dp_pull_demo(3, 2)
print("đáp án (0,0):", table[0][0])

h=0: [1.0, 1.0, 1.0]
h=1: [0.25, 0.25, 0.25] ... (hàng đầu)
h=2: [0.0625, 0.0625, 0.0625] ... (hàng đầu)
đáp án (0,0): 0.0625


In [13]:
class Sol3:
    """Cách 3: DP kéo (pull), mảng cuốn chiếu."""
    def knightProbability(self, n: int, k: int, row: int, column: int) -> float:
        cur = [[1.0] * n for _ in range(n)]
        for _ in range(k):
            nxt = [[0.0] * n for _ in range(n)]
            for i in range(n):
                for j in range(n):
                    total = 0.0
                    for dr, dc in MOVES:
                        x, y = i + dr, j + dc
                        if 0 <= x < n and 0 <= y < n:
                            total += cur[x][y]
                    nxt[i][j] = total / 8
            cur = nxt
        return cur[row][column]

for n, k, r, c, expected in examples:
    assert close(Sol3().knightProbability(n, k, r, c), expected)
print("Sol3 OK")

Sol3 OK


**Độ phức tạp:** Time `O(8 · k · n²)` (≈ `500 000` phép cho `n = 25, k = 100`), Space `O(n²)`.

### 7.3. Cách 4 — Quy hoạch động đẩy (push): phân phối xác suất từ ô xuất phát

Góc nhìn "đi xuôi": gọi `dist[i][j]` là **xác suất mã đang ở ô `(i, j)` và còn trên bàn** sau `h` nước. Ban đầu `dist[row][column] = 1`. Mỗi bước, mỗi ô chia xác suất của nó **đều** cho 8 ô đích: `1/8` cho mỗi ô đích **nếu ô đích nằm trong bàn** (phần ra ngoài bị "rơi mất"). Đáp án là **tổng** `dist` sau `k` bước (xác suất còn trên bàn). Đây là cùng bài toán nhưng đi theo chiều ngược của mũi tên, tương đương nhân vectơ với ma trận chuyển.

In [14]:
def push_demo(n, k, r, c):
    dist = [[0.0] * n for _ in range(n)]
    dist[r][c] = 1.0
    for h in range(1, k + 1):
        nxt = [[0.0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                p = dist[i][j]
                if p == 0.0:
                    continue
                for dr, dc in MOVES:
                    x, y = i + dr, j + dc
                    if 0 <= x < n and 0 <= y < n:
                        nxt[x][y] += p / 8
        dist = nxt
        print(f"h={h}: tổng còn trên bàn = {sum(map(sum, dist))}")
        for row in dist:
            print("   ", [round(v, 4) for v in row])
    return sum(map(sum, dist))

print(push_demo(3, 2, 0, 0))

h=1: tổng còn trên bàn = 0.25
    [0.0, 0.0, 0.0]
    [0.0, 0.0, 0.125]
    [0.0, 0.125, 0.0]
h=2: tổng còn trên bàn = 0.0625
    [0.0312, 0.0, 0.0156]
    [0.0, 0.0, 0.0]
    [0.0156, 0.0, 0.0]
0.0625


In [15]:
class Sol4:
    """Cách 4: DP đẩy (push), phân phối xác suất từ ô xuất phát."""
    def knightProbability(self, n: int, k: int, row: int, column: int) -> float:
        dist = [[0.0] * n for _ in range(n)]
        dist[row][column] = 1.0
        for _ in range(k):
            nxt = [[0.0] * n for _ in range(n)]
            for i in range(n):
                for j in range(n):
                    p = dist[i][j]
                    if p:
                        share = p / 8
                        for dr, dc in MOVES:
                            x, y = i + dr, j + dc
                            if 0 <= x < n and 0 <= y < n:
                                nxt[x][y] += share
            dist = nxt
        return sum(map(sum, dist))

for n, k, r, c, expected in examples:
    assert close(Sol4().knightProbability(n, k, r, c), expected)
print("Sol4 OK")

Sol4 OK


**Độ phức tạp:** Time `O(8 · k · n²)`, Space `O(n²)`. Ưu điểm: ra luôn **phân phối vị trí** của mã sau mỗi bước (hữu ích cho biến thể ở mục 15), và bỏ qua nhanh các ô có xác suất 0.

### 7.4. Cách 5 — Tận dụng đối xứng: chỉ tính các ô "chính tắc"

Bàn `n x n` đối xứng qua 8 phép (4 phép lật/xoay nhóm hai phép phản xạ qua trục dọc/ngang và phản xạ qua đường chéo). Quân mã và luật biên đều bất biến dưới các phép này, nên `f(h, i, j) = f(h, i', j')` với mọi `(i', j')` đối xứng với `(i, j)`.

Ô chính tắc: `i' = min(i, n-1-i)`, `j' = min(j, n-1-j)`, rồi đặt `i' ≤ j'` (đổi chỗ nếu cần). Chỉ còn khoảng `n²/8` ô khác nhau (với `n = 25`: `13 · 14 / 2 = 91` thay vì `625`). Mỗi bước ta tính `f` cho các ô chính tắc; khi cần giá trị của ô kề `(x, y)` ta **quy về ô chính tắc** của nó.

In [16]:
def make_canon(n):
    def canon(i, j):
        i = min(i, n - 1 - i)
        j = min(j, n - 1 - j)
        return (i, j) if i <= j else (j, i)
    return canon

n = 5
canon = make_canon(n)
cells = sorted({canon(i, j) for i in range(n) for j in range(n)})
print("n=5: số ô chính tắc =", len(cells), "thay vì", n * n, "->", cells)
print("canon(4,1) =", canon(4, 1), "| canon(0,3) =", canon(0, 3), "| canon(2,2) =", canon(2, 2))

n=5: số ô chính tắc = 6 thay vì 25 -> [(0, 0), (0, 1), (0, 2), (1, 1), (1, 2), (2, 2)]
canon(4,1) = (0, 1) | canon(0,3) = (0, 1) | canon(2,2) = (2, 2)


In [17]:
class Sol5:
    """Cách 5: DP kéo chỉ trên các ô chính tắc (đối xứng 8 phép)."""
    def knightProbability(self, n: int, k: int, row: int, column: int) -> float:
        def canon(i, j):
            i = min(i, n - 1 - i)
            j = min(j, n - 1 - j)
            return (i, j) if i <= j else (j, i)

        half = (n + 1) // 2
        cells = [(i, j) for i in range(half) for j in range(i, half)]
        f = {cell: 1.0 for cell in cells}
        for _ in range(k):
            g = {}
            for i, j in cells:
                total = 0.0
                for dr, dc in MOVES:
                    x, y = i + dr, j + dc
                    if 0 <= x < n and 0 <= y < n:
                        total += f[canon(x, y)]
                g[(i, j)] = total / 8
            f = g
        return f[canon(row, column)]

for n, k, r, c, expected in examples:
    assert close(Sol5().knightProbability(n, k, r, c), expected)
print("Sol5 OK")

Sol5 OK


**Độ phức tạp:** Time `O(8 · k · n²/8) = O(k · n²)` hằng số nhỏ hơn khoảng 8 lần so với Cách 3, Space `O(n²/8)`. Với đề này (`n ≤ 25`) không cần thiết, nhưng là ví dụ hay về việc dùng đối xứng để giảm không gian trạng thái.

## 8. TRACE từng bước

### 8.1. Trace Cách 3 (DP kéo) trên bàn 3x3, `k = 2`

Bảng trạng thái (xác suất sống sót `h` nước nếu xuất phát từ ô đó; ta in bảng đầy đủ bằng code ngay sau đó):

| ô | `h = 0` | `h = 1` | `h = 2` |
|---|---|---|---|
| `(0,0)` góc | 1 | 2/8 = 0.25 | 0.0625 |
| `(0,1)` cạnh | 1 | 0.25 | 0.0625 |
| `(1,1)` tâm | 1 | 0 | 0 |

Ta in bảng thật bằng code (dưới dạng phân số chính xác) để đối chiếu.

In [18]:
def trace_dp(n, k):
    cur = [[Fraction(1)] * n for _ in range(n)]
    print("h=0:")
    for row in cur:
        print("   ", [str(v) for v in row])
    for h in range(1, k + 1):
        nxt = [[Fraction(0)] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                nxt[i][j] = sum((cur[i + dr][j + dc] for dr, dc in MOVES if 0 <= i + dr < n and 0 <= j + dc < n), Fraction(0)) / 8
        cur = nxt
        print(f"h={h}:")
        for row in cur:
            print("   ", [str(v) for v in row])
    return cur

trace_dp(3, 2)

h=0:
    ['1', '1', '1']
    ['1', '1', '1']
    ['1', '1', '1']
h=1:
    ['1/4', '1/4', '1/4']
    ['1/4', '0', '1/4']
    ['1/4', '1/4', '1/4']
h=2:
    ['1/16', '1/16', '1/16']
    ['1/16', '0', '1/16']
    ['1/16', '1/16', '1/16']


[[Fraction(1, 16), Fraction(1, 16), Fraction(1, 16)],
 [Fraction(1, 16), Fraction(0, 1), Fraction(1, 16)],
 [Fraction(1, 16), Fraction(1, 16), Fraction(1, 16)]]

Đọc bảng: sau `h = 1` mọi ô ngoài tâm có `1/4` (2 nước trong bàn), tâm `0`. Sau `h = 2`: ô góc `(0,0)` có hai ô kề trong bàn là `(1,2)` và `(2,1)`, mỗi ô `1/4` → `(1/4 + 1/4)/8 = 1/16 = 0.0625` ✓. Còn tâm `(1,1)` vẫn là `0` vì không có nước nào ở lại.

### 8.2. Trace Cách 2 (đệ quy có nhớ): ai được gọi, ai dùng lại kết quả

Ta in từng lời gọi `f(h, i, j)` với thụt lề theo độ sâu, đánh dấu "dùng lại cache" khi đã tính.

In [19]:
def trace_memo(n, k, r, c, limit=40):
    seen = set()
    printed = [0]

    def f(h, i, j, depth):
        inside = 0 <= i < n and 0 <= j < n
        key = (h, i, j)
        tag = ""
        if key in seen and inside and h > 0:
            tag = " (dùng lại cache)"
        if printed[0] < limit:
            print("  " * depth + f"f({h}, {i}, {j}){tag}")
            printed[0] += 1
        if not inside:
            return 0.0
        if h == 0:
            return 1.0
        if key in seen:
            return memo[key]
        seen.add(key)
        val = sum(f(h - 1, i + dr, j + dc, depth + 1) for dr, dc in MOVES) / 8
        memo[key] = val
        return val

    memo = {}
    ans = f(k, r, c, 0)
    print("... đáp án =", ans, "| số trạng thái đã tính:", len(seen))
    return ans

trace_memo(3, 2, 0, 0)

f(2, 0, 0)
  f(1, -2, -1)
  f(1, -2, 1)
  f(1, -1, -2)
  f(1, -1, 2)
  f(1, 1, -2)
  f(1, 1, 2)
    f(0, -1, 1)
    f(0, -1, 3)
    f(0, 0, 0)
    f(0, 0, 4)
    f(0, 2, 0)
    f(0, 2, 4)
    f(0, 3, 1)
    f(0, 3, 3)
  f(1, 2, -1)
  f(1, 2, 1)
    f(0, 0, 0)
    f(0, 0, 2)
    f(0, 1, -1)
    f(0, 1, 3)
    f(0, 3, -1)
    f(0, 3, 3)
    f(0, 4, 0)
    f(0, 4, 2)
... đáp án = 0.0625 | số trạng thái đã tính: 3


0.0625

Quan sát: từ `f(2, 0, 0)` ta gọi hai ô trong bàn `(1,2)`, `(2,1)` (6 nước còn lại ra ngoài trả `0` ngay). Ở `n` lớn hơn, nhiều nhánh cùng dẫn về một ô nên cache phát huy tác dụng.

### 8.3. Trace Cách 4 (đẩy): xác suất "đang ở ô" qua từng bước

`push_demo(3, 2, 0, 0)` ở mục 7.3 cho thấy khối xác suất 1 ở `(0,0)` chia thành `1/8` ở `(1,2)` và `1/8` ở `(2,1)` (và `6/8` rơi ra ngoài). Bước sau mỗi khối `1/8` lại chia: `1/64` vào mỗi ô đích trong bàn. Ta kiểm tra "tổng xác suất mất" dần:

In [20]:
def lost_mass(n, k, r, c):
    dist = [[Fraction(0)] * n for _ in range(n)]
    dist[r][c] = Fraction(1)
    for h in range(1, k + 1):
        nxt = [[Fraction(0)] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                for dr, dc in MOVES:
                    x, y = i + dr, j + dc
                    if 0 <= x < n and 0 <= y < n:
                        nxt[x][y] += dist[i][j] / 8
        dist = nxt
        alive = sum(sum(row) for row in dist)
        print(f"sau {h} nước: còn trên bàn = {alive} = {float(alive):.5f}, đã rơi ra ngoài (tích luỹ) = {1 - alive}")

lost_mass(3, 3, 0, 0)
lost_mass(5, 3, 2, 2)

sau 1 nước: còn trên bàn = 1/4 = 0.25000, đã rơi ra ngoài (tích luỹ) = 3/4
sau 2 nước: còn trên bàn = 1/16 = 0.06250, đã rơi ra ngoài (tích luỹ) = 15/16
sau 3 nước: còn trên bàn = 1/64 = 0.01562, đã rơi ra ngoài (tích luỹ) = 63/64
sau 1 nước: còn trên bàn = 1 = 1.00000, đã rơi ra ngoài (tích luỹ) = 0
sau 2 nước: còn trên bàn = 3/8 = 0.37500, đã rơi ra ngoài (tích luỹ) = 5/8
sau 3 nước: còn trên bàn = 1/4 = 0.25000, đã rơi ra ngoài (tích luỹ) = 3/4


### 8.4. Trace Cách 5: các ô chính tắc

Với `n = 5`, có 6 ô chính tắc `(0,0), (0,1), (0,2), (1,1), (1,2), (2,2)`. Ta in giá trị `f` sau mỗi bước cho các ô này và kiểm tra trùng với toàn bộ bảng của Cách 3:

In [21]:
def trace_sym(n, k):
    canon = make_canon(n)
    half = (n + 1) // 2
    cells = [(i, j) for i in range(half) for j in range(i, half)]
    f = {cell: 1.0 for cell in cells}
    print("h=0:", {c: f[c] for c in cells})
    for h in range(1, k + 1):
        g = {}
        for i, j in cells:
            tot = 0.0
            for dr, dc in MOVES:
                x, y = i + dr, j + dc
                if 0 <= x < n and 0 <= y < n:
                    tot += f[canon(x, y)]
            g[(i, j)] = tot / 8
        f = g
        print(f"h={h}:", {c: round(f[c], 5) for c in cells})
    # đối chiếu với Cách 3 trên toàn bảng
    for i in range(n):
        for j in range(n):
            assert close(f[canon(i, j)], Sol3().knightProbability(n, h, i, j))

trace_sym(5, 3)
print("đối xứng khớp toàn bảng")

h=0: {(0, 0): 1.0, (0, 1): 1.0, (0, 2): 1.0, (1, 1): 1.0, (1, 2): 1.0, (2, 2): 1.0}
h=1: {(0, 0): 0.25, (0, 1): 0.375, (0, 2): 0.5, (1, 1): 0.5, (1, 2): 0.75, (2, 2): 1.0}
h=2: {(0, 0): 0.1875, (0, 1): 0.25, (0, 2): 0.28125, (1, 1): 0.28125, (1, 2): 0.3125, (2, 2): 0.375}
h=3: {(0, 0): 0.07812, (0, 1): 0.11719, (0, 2): 0.14062, (1, 1): 0.14062, (1, 2): 0.1875, (2, 2): 0.25}
đối xứng khớp toàn bảng


## 9. Code nộp LeetCode

Dán đoạn dưới vào LeetCode (đã có sẵn `List`; ở bài này ta chỉ dùng thư viện chuẩn là không cần import gì thêm).

In [22]:
class Solution:
    def knightProbability(self, n: int, k: int, row: int, column: int) -> float:
        moves = [(-2, -1), (-2, 1), (-1, -2), (-1, 2), (1, -2), (1, 2), (2, -1), (2, 1)]
        # cur[i][j] = xác suất còn trên bàn sau h nước nếu xuất phát từ (i, j); h = 0 -> 1
        cur = [[1.0] * n for _ in range(n)]
        for _ in range(k):
            nxt = [[0.0] * n for _ in range(n)]
            for i in range(n):
                for j in range(n):
                    total = 0.0
                    for dr, dc in moves:
                        x, y = i + dr, j + dc
                        if 0 <= x < n and 0 <= y < n:
                            total += cur[x][y]
                    nxt[i][j] = total / 8
            cur = nxt
        return cur[row][column]


for n, k, r, c, expected in examples:
    got = Solution().knightProbability(n, k, r, c)
    print(f"{got:.5f}", expected)
    assert close(got, expected)
print("Solution chạy đúng 5 ví dụ")

0.06250 0.0625
1.00000 1.0
0.00000 0.0
0.00000 0.0
0.50000 0.5
Solution chạy đúng 5 ví dụ


## 10. Vì sao thuật toán đúng

Gọi `P_h(u)` là xác suất (thật sự, theo mô hình của đề) để mã còn trên bàn sau `h` nước nếu bắt đầu từ ô `u` **trong bàn** (và đặt `P_h(u) = 0` nếu `u` ngoài bàn).

**Mô hình.** Mỗi lượt chọn đều trong 8 nước, các lượt độc lập. Mã "còn trên bàn khi dừng" nghĩa là **cả `k` nước đều không đưa mã ra ngoài** (nếu ra ngoài thì dừng và mất).

**Đẳng thức.** Với `h ≥ 1`, điều kiện trên chia theo nước đầu tiên `m` (xác suất `1/8` mỗi nước): mã sống sót `h` nước ⇔ nước đầu đưa tới ô `v = u + m` **trong bàn** và từ `v` còn sống sót tiếp `h - 1` nước. Do các nước đầu loại trừ nhau và các lượt sau độc lập với lượt đầu (chỉ phụ thuộc vị trí mới):

`P_h(u) = Σ_m (1/8) · P_{h-1}(u + m)`

(với quy ước ô ngoài bàn có `P = 0`, nên các nước ra ngoài đóng góp 0). Và `P_0(u) = 1` với `u` trong bàn. Đây chính là hệ thức của Cách 2 và Cách 3, nên chúng cho `P_k(row, column)`. (Quy nạp theo `h` cho tính đúng.)

**Cách 4 (đẩy)** đúng vì `dist_h[v]` = xác suất mã ở ô `v` sau `h` nước mà chưa từng ra ngoài; công thức `dist_h[v] = Σ_{u kề v} dist_{h-1}[u]/8` (chỉ cộng từ các ô `u` trong bàn) đúng theo định nghĩa, và `Σ_v dist_k[v]` là xác suất còn trên bàn. Hai cách (kéo và đẩy) là hai cách tính cùng một đại lượng `e_{(row,col)}^T · A^k · 1` với `A` là ma trận chuyển (đã bỏ phần ra ngoài): kéo nhân từ bên phải, đẩy nhân từ bên trái. Vì ma trận chuyển `A` **đối xứng** (nước đi của mã có tính hai chiều: nếu `u → v` thì `v → u`) nên hai cách thậm chí cho **cùng bảng**.

**Cách 5 (đối xứng)** đúng vì nhóm 8 phép đối xứng của hình vuông bảo toàn tập nước đi (`MOVES` đóng kín dưới các phép) và bảo toàn "trong bàn", nên `P_h` là hàm hằng trên mỗi lớp đối xứng.

**Cách 1** đúng theo định nghĩa: `P_k = (số chuỗi k nước luôn trong bàn) / 8^k`.

**Về sai số số thực:** mỗi phép `+` và `/ 8` sai số tương đối `≤ 2^-53`; chia cho 8 là **chính xác** (luỹ thừa của 2), chỉ phép cộng gây sai số. Tổng sai số sau `k · 8` bước cộng nhỏ hơn `10^-12` — nhỏ hơn rất nhiều ngưỡng `10^-5` của LeetCode. Ta sẽ kiểm tra bằng `Fraction` ở mục 12.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê mọi đường đi | `O(8^k)` | `O(k)` |
| 2. Đệ quy có nhớ | `O(k · n²)` | `O(k · n²)` |
| 3. DP kéo cuốn chiếu | `O(k · n²)` | `O(n²)` |
| 4. DP đẩy | `O(k · n²)` | `O(n²)` |
| 5. Đối xứng | `O(k · n²)` (hằng số nhỏ ≈ 1/8) | `O(n²/8)` |

Với `n = 25`, `k = 100`: `8 · 100 · 625 = 500 000` phép cộng — chạy trong khoảng nửa giây ở Python. Đo thực nghiệm:

In [23]:
n, k = 25, 100
for name, cls in [("Cách 2", Sol2), ("Cách 3", Sol3), ("Cách 4", Sol4), ("Cách 5", Sol5), ("Solution", Solution)]:
    t0 = time.perf_counter()
    res = cls().knightProbability(n, k, 12, 12)
    print(f"{name:<9}: {1000 * (time.perf_counter() - t0):8.2f} ms -> {res:.10f}")

Cách 2   :    97.72 ms -> 0.0474287607


Cách 3   :    76.67 ms -> 0.0474287607


Cách 4   :    41.40 ms -> 0.0474287607
Cách 5   :    38.43 ms -> 0.0474287607


Solution :    72.83 ms -> 0.0474287607


## 12. Bộ test

### 12.1. Ví dụ đề và edge case

**Oracle độc lập:** đếm đường đi **nguyên** bằng đệ quy thuần (`knight_brute_count`), rồi xác suất chính xác `Fraction(count, 8^k)` — không dùng bảng DP, nhớ hay phân phối. Vì kết quả là số thực, mọi so sánh dùng `close(a, b)` với sai số `1e-9` (chặt hơn nhiều so với `1e-5` của LeetCode). Ta gom các cách vào dict `solvers`.

In [24]:
def oracle(n, k, r, c):
    return Fraction(knight_brute_count(n, k, r, c), 8 ** k)

solvers = {
    "brute": knight_brute,
    "memo": lambda n, k, r, c: Sol2().knightProbability(n, k, r, c),
    "pull": lambda n, k, r, c: Sol3().knightProbability(n, k, r, c),
    "push": lambda n, k, r, c: Sol4().knightProbability(n, k, r, c),
    "symmetry": lambda n, k, r, c: Sol5().knightProbability(n, k, r, c),
    "leetcode": lambda n, k, r, c: Solution().knightProbability(n, k, r, c),
}

for n, k, r, c, expected in examples:
    assert close(float(oracle(n, k, r, c)), expected)
    for name, fn in solvers.items():
        assert close(fn(n, k, r, c), expected), (name, n, k, r, c)
print("Ví dụ đề: OK cho cả 6 solver và oracle (so sánh có sai số)")

Ví dụ đề: OK cho cả 6 solver và oracle (so sánh có sai số)


In [25]:
edge_cases = [
    (1, 0, 0, 0, 1.0, "bàn 1 ô, không đi"),
    (1, 5, 0, 0, 0.0, "bàn 1 ô, đi là ra ngoài"),
    (2, 1, 0, 0, 0.0, "bàn 2x2: mọi nước ra ngoài"),
    (3, 1, 1, 1, 0.0, "tâm bàn 3x3"),
    (3, 3, 1, 1, 0.0, "tâm bàn 3x3, nhiều nước"),
    (3, 0, 1, 1, 1.0, "k = 0 ở bàn bất kỳ"),
    (4, 0, 2, 3, 1.0, "k = 0"),
    (4, 1, 1, 1, 0.5, "4 trong 8 nước trong bàn"),
    (5, 1, 2, 2, 1.0, "tâm bàn 5x5: cả 8 nước trong bàn"),
    (8, 1, 0, 0, 2 / 8, "góc bàn 8x8: 2 nước"),
    (8, 1, 3, 3, 1.0, "giữa bàn 8x8: cả 8 nước"),
    (3, 2, 0, 0, 0.0625, "ví dụ 1"),
]
for n, k, r, c, expected, note in edge_cases:
    assert close(float(oracle(n, k, r, c)), expected), ("oracle", note)
    for name, fn in solvers.items():
        assert close(fn(n, k, r, c), expected), (name, note)
print(f"{len(edge_cases)} edge case: OK")

# ô đối xứng phải cho cùng xác suất
for n, k in [(5, 4), (6, 3), (7, 5)]:
    for r, c in product(range(n), repeat=2):
        base = Sol3().knightProbability(n, k, r, c)
        for rr, cc in [(c, r), (n - 1 - r, c), (r, n - 1 - c), (n - 1 - r, n - 1 - c)]:
            assert close(base, Sol3().knightProbability(n, k, rr, cc))
print("tính đối xứng của đáp án: OK")

12 edge case: OK


tính đối xứng của đáp án: OK


### 12.2. Phản ví dụ cho các ý tưởng sai

- **Chia cho số nước hợp lệ** (thay vì 8): nghĩ rằng "mã chỉ chọn trong các nước còn trong bàn" — sai, đề nói mã chọn đều trong 8 nước kể cả ra ngoài.
- **Tưởng xác suất chỉ là tỉ lệ ô trong bàn mỗi bước nhân lên** (độc lập): `P_k = p^k` với `p` = xác suất ở bước đầu — sai vì xác suất phụ thuộc ô hiện tại.
- **So sánh số thực bằng `==`** — không đáng tin do sai số làm tròn.

In [26]:
def wrong_divide_valid(n, k, r, c):
    cur = [[1.0] * n for _ in range(n)]
    for _ in range(k):
        nxt = [[0.0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                nbrs = [(i + dr, j + dc) for dr, dc in MOVES if 0 <= i + dr < n and 0 <= j + dc < n]
                if nbrs:
                    nxt[i][j] = sum(cur[x][y] for x, y in nbrs) / len(nbrs)   # sai: chia số nước hợp lệ
        cur = nxt
    return cur[r][c]

def wrong_independent(n, k, r, c):
    p = sum(1 for dr, dc in MOVES if 0 <= r + dr < n and 0 <= c + dc < n) / 8
    return p ** k                                                           # sai: coi các bước độc lập y hệt

for n, k, r, c in [(3, 2, 0, 0), (4, 2, 1, 1), (5, 3, 0, 0)]:
    truth = float(oracle(n, k, r, c))
    print(f"n={n} k={k} ({r},{c}) | đúng={truth:.6f} | chia hợp lệ={wrong_divide_valid(n, k, r, c):.6f} | độc lập={wrong_independent(n, k, r, c):.6f}")
assert not close(wrong_divide_valid(3, 2, 0, 0), 0.0625)
assert not close(wrong_independent(4, 2, 1, 1), float(oracle(4, 2, 1, 1)))
print("0.1 + 0.2 == 0.3 ->", 0.1 + 0.2 == 0.3, "| nhưng close ->", close(0.1 + 0.2, 0.3))

n=3 k=2 (0,0) | đúng=0.062500 | chia hợp lệ=1.000000 | độc lập=0.062500
n=4 k=2 (1,1) | đúng=0.156250 | chia hợp lệ=1.000000 | độc lập=0.250000
n=5 k=3 (0,0) | đúng=0.078125 | chia hợp lệ=1.000000 | độc lập=0.015625
0.1 + 0.2 == 0.3 -> False | nhưng close -> True


### 12.3. Đối chiếu mọi cách bằng vét cạn toàn bộ input nhỏ

Duyệt **mọi** `n = 1..5`, mọi ô `(row, column)`, mọi `k = 0..5` (với `n ≥ 4` giới hạn `k ≤ 4` để đường vét cạn không quá chậm). So 6 solver với oracle `Fraction`.

In [27]:
checked = 0
for n in range(1, 6):
    kmax = 5 if n <= 3 else 4
    for r, c in product(range(n), repeat=2):
        for k in range(kmax + 1):
            exact = float(oracle(n, k, r, c))
            for name, fn in solvers.items():
                assert close(fn(n, k, r, c), exact), (name, n, k, r, c)
            checked += 1
print("đã kiểm", checked, "bộ (n, k, row, column): tất cả khớp oracle trong sai số 1e-9")

đã kiểm 289 bộ (n, k, row, column): tất cả khớp oracle trong sai số 1e-9


### 12.4. Test ngẫu nhiên có seed (đúng ràng buộc) và đối chiếu chính xác bằng số nguyên

Với `n` và `k` lớn, oracle vét cạn không dùng được. Ta dùng một **bộ đếm nguyên độc lập**: đếm số đường đi hợp lệ bằng DP trên số nguyên (không có số thực), rồi `Fraction(count, 8^k)`. Đây là so sánh chính xác để ước lượng sai số thực sự của các cách dùng `float`.

In [28]:
def exact_integer(n, k, r, c):
    ways = [[0] * n for _ in range(n)]
    ways[r][c] = 1
    for _ in range(k):
        nxt = [[0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                w = ways[i][j]
                if w:
                    for dr, dc in MOVES:
                        x, y = i + dr, j + dc
                        if 0 <= x < n and 0 <= y < n:
                            nxt[x][y] += w
        ways = nxt
    return Fraction(sum(map(sum, ways)), 8 ** k)

# kiểm chứng bộ đếm nguyên bằng oracle vét cạn trên input nhỏ
for n, k, r, c in [(3, 2, 0, 0), (5, 4, 2, 2), (6, 3, 1, 4), (8, 4, 0, 7)]:
    assert exact_integer(n, k, r, c) == oracle(n, k, r, c)

rng = random.Random(688)
worst = 0.0
for _ in range(300):
    n = rng.randint(1, 25)
    k = rng.randint(0, 100)
    r, c = rng.randrange(n), rng.randrange(n)
    assert 1 <= n <= 25 and 0 <= k <= 100 and 0 <= r < n and 0 <= c < n
    exact = float(exact_integer(n, k, r, c))
    assert 0.0 <= exact <= 1.0
    for name in ["memo", "pull", "push", "symmetry", "leetcode"]:
        err = abs(solvers[name](n, k, r, c) - exact)
        worst = max(worst, err)
        assert err <= 1e-12, (name, n, k, r, c, err)
print("300 test ngẫu nhiên lớn: OK; sai số tuyệt đối lớn nhất so với đáp án chính xác =", worst)

300 test ngẫu nhiên lớn: OK; sai số tuyệt đối lớn nhất so với đáp án chính xác = 1.1102230246251565e-16


### 12.5. Tính chất của đáp án và test hiệu năng

Ngoài việc so với đáp án chính xác, đáp án phải thoả các **tính chất**: (a) nằm trong `[0, 1]`; (b) **không tăng** theo `k` (đi thêm nước không thể làm xác suất sống sót tăng); (c) `k = 0` luôn bằng `1`; (d) xác suất sau `k` nước từ ô `u` bằng tổng `dist` của Cách 4 (hai chiều kéo/đẩy khớp nhau).

In [29]:
rng = random.Random(6880)
for _ in range(120):
    n = rng.randint(1, 25)
    r, c = rng.randrange(n), rng.randrange(n)
    prev = 1.0
    for k in range(0, 13):
        p = Solution().knightProbability(n, k, r, c)
        assert 0.0 <= p <= 1.0 + 1e-12
        assert p <= prev + 1e-12, (n, k, r, c)
        prev = p
    assert Solution().knightProbability(n, 0, r, c) == 1.0

# trường hợp lớn nhất của đề
t0 = time.perf_counter()
big = {name: solvers[name](25, 100, 12, 12) for name in ["memo", "pull", "push", "symmetry", "leetcode"]}
print("n=25,k=100 tại tâm:", {k_: round(v, 12) for k_, v in big.items()}, f"| {time.perf_counter() - t0:.2f}s")
vals = list(big.values())
assert max(vals) - min(vals) < 1e-12
# bàn nhỏ, k lớn: xác suất cực nhỏ nhưng không âm/không lỗi
tiny = Solution().knightProbability(3, 100, 0, 0)
assert 0.0 <= tiny < 1e-50
print("n=3,k=100 từ góc:", tiny)

n=25,k=100 tại tâm: {'memo': 0.047428760694, 'pull': 0.047428760694, 'push': 0.047428760694, 'symmetry': 0.047428760694, 'leetcode': 0.047428760694} | 0.29s
n=3,k=100 từ góc: 6.223015277861142e-61


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chia cho số nước hợp lệ thay vì chia cho 8

**Code sai:**

In [30]:
class WrongSolution1:
    def knightProbability(self, n, k, row, column):
        cur = [[1.0] * n for _ in range(n)]
        for _ in range(k):
            nxt = [[0.0] * n for _ in range(n)]
            for i in range(n):
                for j in range(n):
                    nb = [(i + dr, j + dc) for dr, dc in MOVES if 0 <= i + dr < n and 0 <= j + dc < n]
                    if nb:
                        nxt[i][j] = sum(cur[x][y] for x, y in nb) / len(nb)     # sai
            cur = nxt
        return cur[row][column]

print("sai  :", WrongSolution1().knightProbability(3, 2, 0, 0), "(đúng phải là 0.0625)")
print("đúng :", Solution().knightProbability(3, 2, 0, 0))

sai  : 1.0 (đúng phải là 0.0625)
đúng : 0.0625


**Vì sao sai:** đề nói mã chọn **đều** trong 8 nước, kể cả nước đưa ra ngoài. Nước ra ngoài **vẫn có xác suất `1/8`** và dẫn tới thất bại. Chia cho `len(nb)` coi như mã "biết tránh" biên, làm xác suất sai. **Code đúng:** luôn chia cho `8`.

### Lỗi 2: Quên trường hợp `k = 0` (hoặc khởi tạo sai tầng 0)

In [31]:
class WrongSolution2:
    def knightProbability(self, n, k, row, column):
        cur = [[0.0] * n for _ in range(n)]            # sai: tầng 0 phải là 1.0 cho mọi ô trong bàn
        for _ in range(k):
            nxt = [[0.0] * n for _ in range(n)]
            for i in range(n):
                for j in range(n):
                    nxt[i][j] = sum(cur[i + dr][j + dc] for dr, dc in MOVES if 0 <= i + dr < n and 0 <= j + dc < n) / 8
            cur = nxt
        return cur[row][column]

print("k=0 sai :", WrongSolution2().knightProbability(1, 0, 0, 0), "| đúng:", Solution().knightProbability(1, 0, 0, 0))
print("k=2 sai :", WrongSolution2().knightProbability(3, 2, 0, 0), "| đúng:", Solution().knightProbability(3, 2, 0, 0))

k=0 sai : 0.0 | đúng: 1.0
k=2 sai : 0.0 | đúng: 0.0625


**Vì sao sai:** điều kiện biên: không đi nước nào thì mã chắc chắn còn trên bàn: `f(0, ô trong bàn) = 1`. Khởi tạo `0` làm mọi đáp án thành `0`. **Code đúng:** tầng 0 toàn `1.0`.

### Lỗi 3: Tạo bảng bằng `[[0] * n] * n` (các hàng dùng chung)

In [32]:
class WrongSolution3:
    def knightProbability(self, n, k, row, column):
        cur = [[1.0] * n] * n                       # sai: n tham chiếu tới cùng một hàng
        for _ in range(k):
            nxt = [[0.0] * n] * n                   # sai tương tự
            for i in range(n):
                for j in range(n):
                    total = 0.0
                    for dr, dc in MOVES:
                        x, y = i + dr, j + dc
                        if 0 <= x < n and 0 <= y < n:
                            total += cur[x][y]
                    nxt[i][j] = total / 8           # ghi vào ô (i, j) làm đổi cả cột j ở mọi hàng
            cur = nxt
        return cur[row][column]

for n, k, r, c in [(3, 2, 0, 0), (5, 3, 0, 0)]:
    print(f"n={n} k={k} | sai: {WrongSolution3().knightProbability(n, k, r, c):.6f} | đúng: {Solution().knightProbability(n, k, r, c):.6f}")

n=3 k=2 | sai: 0.062500 | đúng: 0.062500
n=5 k=3 | sai: 0.037109 | đúng: 0.078125


**Vì sao sai:** mọi hàng của `nxt` là **cùng một list**, nên gán `nxt[i][j]` ghi đè lên cột `j` của tất cả các hàng. **Code đúng:** `[[0.0] * n for _ in range(n)]`.

### Lỗi 4: Đệ quy không nhớ (quá thời gian)

In [33]:
def wrong_no_memo(n, k, r, c):
    def f(h, i, j):
        if not (0 <= i < n and 0 <= j < n):
            return 0.0
        if h == 0:
            return 1.0
        return sum(f(h - 1, i + dr, j + dc) for dr, dc in MOVES) / 8
    return f(k, r, c)

for k in (2, 4, 6, 7):
    t0 = time.perf_counter()
    v = wrong_no_memo(8, k, 3, 3)
    t1 = time.perf_counter() - t0
    assert close(v, Solution().knightProbability(8, k, 3, 3))
    print(f"k={k}: {1000 * t1:9.2f} ms (kết quả đúng nhưng thời gian ×8 mỗi bước; k = 100 là không thể)")

k=2:      0.04 ms (kết quả đúng nhưng thời gian ×8 mỗi bước; k = 100 là không thể)
k=4:      1.10 ms (kết quả đúng nhưng thời gian ×8 mỗi bước; k = 100 là không thể)
k=6:     39.92 ms (kết quả đúng nhưng thời gian ×8 mỗi bước; k = 100 là không thể)


k=7:    170.18 ms (kết quả đúng nhưng thời gian ×8 mỗi bước; k = 100 là không thể)


**Vì sao sai (về hiệu năng):** kết quả đúng nhưng thời gian `O(8^k)`. Thêm `@lru_cache` (hoặc bảng DP) để mỗi `(h, i, j)` chỉ tính một lần.

### Lỗi 5: So sánh số thực bằng `==`

In [34]:
a = Solution().knightProbability(3, 2, 0, 0)
print(a, 4 / 64, a == 4 / 64)                          # ví dụ nhỏ: tình cờ đúng (chia cho 8 là chính xác)
found = None
for n, k in product(range(4, 12), range(2, 30)):
    x = Solution().knightProbability(n, k, 1, 2)
    y = float(exact_integer(n, k, 1, 2))
    if x != y:
        found = (n, k, x, y)
        break
print("tìm thấy ca == thất bại:", found)
if found:
    n, k, x, y = found
    print("|x - y| =", abs(x - y), "| close:", close(x, y))
assert found is not None and close(found[2], found[3])

0.0625 0.0625 True
tìm thấy ca == thất bại: (6, 24, 1.3662240870661567e-05, 1.3662240870661569e-05)
|x - y| = 1.6940658945086007e-21 | close: True


**Vì sao sai:** số thực có sai số làm tròn; hai cách tính "cùng giá trị" có thể khác nhau ở chữ số cuối. **Code đúng:** so sánh bằng `abs(a - b) <= tol` (LeetCode dùng `1e-5`), hoặc dùng `Fraction` khi cần đối chiếu chính xác.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"xác suất sau `k` bước"** trên đồ thị/lưới với chuyển động ngẫu nhiên **không nhớ** → **quy hoạch động theo số bước** (`dp[h][ô]`), thường kéo từ tầng `h - 1`.
- Có hai chiều viết: **kéo** (`f(h, u)` = trung bình `f(h-1, v)` các ô kề) và **đẩy** (chia xác suất của `u` cho các ô đích). Chọn cách nào tuỳ câu hỏi cần "xác suất bắt đầu từ `u`" hay "phân phối đích".
- **Ô ngoài bàn đóng góp 0** (hoặc là trạng thái hấp thụ "đã mất") — nhưng **vẫn chiếm phần xác suất `1/8`** khi chia.
- Đầu ra là số thực → viết test bằng **sai số**; kiểm chứng bằng `Fraction`/đếm số nguyên `ways / 8^k`.
- Bài **đếm đường đi / xác suất có hằng số `8^k`** gợi ý: số nguyên lớn của Python xử lý được `8^100` mà không tràn.
- Dùng **đối xứng** của bàn cờ để giảm trạng thái (chỉ cần khi `n` rất lớn).
- Mảng cuốn chiếu: `cur`/`nxt` — nhớ **tạo `nxt` mới mỗi bước** (hoặc xoá về 0), không cập nhật tại chỗ.
- Luôn thử: `n = 1`, `k = 0`, tâm bàn nhỏ, góc bàn, `k` lớn với bàn nhỏ (xác suất rất bé nhưng không phải 0).

## 15. Biến thể

### 15.1. Số đường đi nguyên (đếm, không chia)

Trả về **số chuỗi `k` nước** mà mã luôn ở trong bàn. Tương tự Cách 3 nhưng dùng số nguyên (Python không tràn).

In [35]:
def ways_stay(n, k, r, c):
    ways = [[1] * n for _ in range(n)]            # kéo: tầng 0 = 1 đường (không đi)
    for _ in range(k):
        nxt = [[0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                nxt[i][j] = sum(ways[i + dr][j + dc] for dr, dc in MOVES if 0 <= i + dr < n and 0 <= j + dc < n)
        ways = nxt
    return ways[r][c]

print(ways_stay(3, 2, 0, 0), ways_stay(8, 3, 0, 0), ways_stay(1, 0, 0, 0))
rng = random.Random(151)
for _ in range(300):
    n = rng.randint(1, 6)
    k = rng.randint(0, 4)
    r, c = rng.randrange(n), rng.randrange(n)
    assert ways_stay(n, k, r, c) == knight_brute_count(n, k, r, c)
print("ways_stay khớp đếm vét cạn")

4 64 1
ways_stay khớp đếm vét cạn


### 15.2. Phân phối vị trí sau `k` nước (từ Cách 4)

Cách 4 cho sẵn `dist` — xác suất mã ở **từng ô** sau `k` nước (và còn trên bàn). Tổng của nó là đáp án. Ta viết hàm trả về bảng đó (số thực) và kiểm tra bằng `Fraction`.

In [36]:
def distribution(n, k, r, c):
    dist = [[0.0] * n for _ in range(n)]
    dist[r][c] = 1.0
    for _ in range(k):
        nxt = [[0.0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                if dist[i][j]:
                    for dr, dc in MOVES:
                        x, y = i + dr, j + dc
                        if 0 <= x < n and 0 <= y < n:
                            nxt[x][y] += dist[i][j] / 8
        dist = nxt
    return dist

d = distribution(5, 2, 2, 2)
for row in d:
    print([round(v, 4) for v in row])
print("tổng =", sum(map(sum, d)), "| đáp án =", Solution().knightProbability(5, 2, 2, 2))
assert close(sum(map(sum, d)), Solution().knightProbability(5, 2, 2, 2))

[0.0, 0.0, 0.0312, 0.0, 0.0]
[0.0, 0.0312, 0.0, 0.0312, 0.0]
[0.0312, 0.0, 0.125, 0.0, 0.0312]
[0.0, 0.0312, 0.0, 0.0312, 0.0]
[0.0, 0.0, 0.0312, 0.0, 0.0]
tổng = 0.375 | đáp án = 0.375


### 15.3. Xác suất mã rơi ra ngoài **đúng ở nước thứ `t`** (và biến thể "xác suất sống sót theo từng bước")

`survival[t]` là xác suất còn trên bàn sau `t` nước, `t = 0..K`. Xác suất rơi ra ngoài đúng ở nước `t` là `survival[t-1] - survival[t]`. Dùng Cách 4 và ghi tổng mỗi bước.

In [37]:
def survival_curve(n, K, r, c):
    dist = [[0.0] * n for _ in range(n)]
    dist[r][c] = 1.0
    out = [1.0]
    for _ in range(K):
        nxt = [[0.0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                if dist[i][j]:
                    for dr, dc in MOVES:
                        x, y = i + dr, j + dc
                        if 0 <= x < n and 0 <= y < n:
                            nxt[x][y] += dist[i][j] / 8
        dist = nxt
        out.append(sum(map(sum, dist)))
    return out

curve = survival_curve(5, 6, 0, 0)
print([round(v, 5) for v in curve])
leave_at = [curve[t - 1] - curve[t] for t in range(1, len(curve))]
print("rời ở nước t:", [round(v, 5) for v in leave_at], "| tổng với còn lại =", sum(leave_at) + curve[-1])
for t in range(len(curve)):
    assert close(curve[t], Solution().knightProbability(5, t, 0, 0))
assert close(sum(leave_at) + curve[-1], 1.0)
print("survival_curve khớp Solution cho từng k")

[1.0, 0.25, 0.1875, 0.07812, 0.04688, 0.02246, 0.01245]
rời ở nước t: [0.75, 0.0625, 0.10938, 0.03125, 0.02441, 0.01001] | tổng với còn lại = 1.0
survival_curve khớp Solution cho từng k


### 15.4. Các bài liên quan trên LeetCode (đọc thêm)

- **576. Out of Boundary Paths:** đếm số đường đi (mod `10^9 + 7`) để quân **ra khỏi** lưới đúng `≤ maxMove` bước — cùng khung DP.
- **935. Knight Dialer:** quân mã trên bàn phím điện thoại, đếm số số điện thoại độ dài `n` (DP theo bước, đồ thị 10 nút).
- **688** (bài này) so với **576**: một bài tính xác suất "ở lại", một bài đếm "thoát ra".
- **62. Unique Paths, 63, 64:** DP lưới cơ bản.
- **837. New 21 Game:** xác suất bằng DP.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê đường đi | duyệt `8^k` chuỗi, đếm hợp lệ | `O(8^k)` | `O(k)` | chỉ để hiểu đề / oracle với `k ≤ 6` |
| 2. Đệ quy có nhớ | `f(h, i, j)` + `lru_cache` | `O(k n²)` | `O(k n²)` | viết nhanh từ định nghĩa |
| 3. DP kéo cuốn chiếu | bảng `cur → nxt` từ `h=0` lên `k` | `O(k n²)` | `O(n²)` | **cách nộp bài chuẩn** |
| 4. DP đẩy | chia xác suất từ ô xuất phát | `O(k n²)` | `O(n²)` | cần phân phối vị trí / xác suất rời ở từng bước |
| 5. Đối xứng | chỉ tính ô chính tắc (≈1/8) | `O(k n²)` (hằng nhỏ) | `O(n²/8)` | `n` rất lớn, muốn giảm trạng thái |

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy (làm tròn 5 chữ số thập phân):

In [38]:
print(round(Solution().knightProbability(3, 0, 1, 1), 5))
print(round(Solution().knightProbability(5, 1, 2, 2), 5))
print(round(Solution().knightProbability(4, 1, 0, 0), 5))
print(round(Solution().knightProbability(2, 3, 0, 1), 5))

1.0
1.0
0.25
0.0


**Bài 2 (tính tay).** Bàn `4 x 4`, mã ở ô `(1, 1)`, `k = 1`: đếm số nước nằm trong bàn rồi tính xác suất. Gán xác suất (số thực) vào `ans2` (đang là `None`).

In [39]:
ans2 = None      # TODO: số thực

**Bài 3 (điền TODO).** Hoàn thành hàm `step(cur, n)`: nhận bảng `cur` (xác suất sống sót `h-1` nước từ mỗi ô) và trả bảng mới `nxt` (xác suất sống sót `h` nước). Dùng `MOVES`, chia cho `8`.

In [40]:
def step(cur, n):
    nxt = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            # TODO: cộng cur[x][y] của các ô kề (x, y) nằm trong bàn rồi chia 8
            pass
    return nxt

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_knight(n, k, row, column)` bằng **đệ quy có nhớ** (`lru_cache`) — Cách 2.

In [41]:
def my_knight(n, k, row, column):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) `knight_exact(n, k, row, column)`: trả về xác suất **chính xác** dạng `Fraction` (gợi ý: đếm số đường đi nguyên rồi chia `8**k`). (b) `best_start(n, k)`: trả về ô `(row, column)` có xác suất sống sót lớn nhất (nếu nhiều ô bằng nhau, chọn ô có `(row, column)` nhỏ nhất theo thứ tự từ điển). Ví dụ `best_start(5, 1)` là `(2, 2)`.

In [42]:
def knight_exact(n, k, row, column):
    # TODO
    pass

def best_start(n, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: `k = 0` thì xác suất 1; đếm số nước trong bàn rồi chia 8.
- Bài 2: liệt kê 8 nước từ `(1, 1)` trên bàn `4 x 4`: ô nào có chỉ số `0..3`?
- Bài 3: giống thân vòng lặp của Cách 3.
- Bài 4: `f(h, i, j)`: ngoài bàn `0.0`, `h == 0` thì `1.0`.
- Bài 5: số đường đi nguyên: DP với số nguyên; so sánh `Fraction`.

**Mức 2 (vừa):**
- Bài 2: từ `(1,1)`: `(3,2)`, `(3,0)`, `(2,3)`, `(0,3)` trong bàn; `(-1,·)` và `(·,-1)` ngoài bàn → 4 nước → `4/8`.
- Bài 3: `total = sum(cur[i + dr][j + dc] for dr, dc in MOVES if 0 <= i + dr < n and 0 <= j + dc < n)`; `nxt[i][j] = total / 8`.
- Bài 4: nhớ dùng `@lru_cache(maxsize=None)` ngay trên hàm đệ quy bên trong.
- Bài 5(a): `Fraction(ways, 8 ** k)`; (b): duyệt mọi ô, `max` với khoá `(xác suất, -...)` hoặc duyệt theo thứ tự và chỉ thay khi **lớn hơn hẳn** (có dung sai nếu dùng số thực; tốt nhất so bằng `Fraction`).

**Mức 3 (gần đáp án):**
- Bài 3: xem Cách 3 ở mục 7.2.
- Bài 4: xem Cách 2 ở mục 7.1.
- Bài 5(b): so sánh bằng `Fraction` để tránh sai số; duyệt `for r in range(n): for c in range(n)` và chỉ cập nhật khi `p > best`.

## 19. Đáp án tham khảo

In [43]:
# Bài 1: 1.0, 1.0, 0.25, 0.0
assert close(Solution().knightProbability(3, 0, 1, 1), 1.0)
assert close(Solution().knightProbability(5, 1, 2, 2), 1.0)
assert close(Solution().knightProbability(4, 1, 0, 0), 0.25)
assert close(Solution().knightProbability(2, 3, 0, 1), 0.0)
print("Bài 1 OK: 1.0, 1.0, 0.25, 0.0")

# Bài 2
ans2_ref = 0.5
assert close(ans2_ref, Solution().knightProbability(4, 1, 1, 1))
print("Bài 2 OK:", ans2_ref)

Bài 1 OK: 1.0, 1.0, 0.25, 0.0
Bài 2 OK: 0.5


In [44]:
# Bài 3
def step_ref(cur, n):
    nxt = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            total = sum(cur[i + dr][j + dc] for dr, dc in MOVES if 0 <= i + dr < n and 0 <= j + dc < n)
            nxt[i][j] = total / 8
    return nxt

for n, k in [(3, 2), (5, 4), (8, 5)]:
    cur = [[1.0] * n for _ in range(n)]
    for _ in range(k):
        cur = step_ref(cur, n)
    for r, c in product(range(n), repeat=2):
        assert close(cur[r][c], Solution().knightProbability(n, k, r, c))
print("Bài 3 OK")

# Bài 4
def my_knight_ref(n, k, row, column):
    @lru_cache(maxsize=None)
    def f(h, i, j):
        if not (0 <= i < n and 0 <= j < n):
            return 0.0
        if h == 0:
            return 1.0
        return sum(f(h - 1, i + dr, j + dc) for dr, dc in MOVES) / 8
    return f(k, row, column)

rng = random.Random(1904)
for _ in range(200):
    n = rng.randint(1, 12)
    k = rng.randint(0, 30)
    r, c = rng.randrange(n), rng.randrange(n)
    assert close(my_knight_ref(n, k, r, c), Solution().knightProbability(n, k, r, c))
print("Bài 4 OK")

Bài 3 OK


Bài 4 OK


In [45]:
# Bài 5
def knight_exact_ref(n, k, row, column):
    ways = [[0] * n for _ in range(n)]
    ways[row][column] = 1
    for _ in range(k):
        nxt = [[0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                if ways[i][j]:
                    for dr, dc in MOVES:
                        x, y = i + dr, j + dc
                        if 0 <= x < n and 0 <= y < n:
                            nxt[x][y] += ways[i][j]
        ways = nxt
    return Fraction(sum(map(sum, ways)), 8 ** k)

def best_start_ref(n, k):
    best, arg = Fraction(-1), None
    for r in range(n):
        for c in range(n):
            p = knight_exact_ref(n, k, r, c)
            if p > best:
                best, arg = p, (r, c)
    return arg

assert knight_exact_ref(3, 2, 0, 0) == Fraction(1, 16)
assert best_start_ref(5, 1) == (2, 2)
rng = random.Random(1905)
for _ in range(100):
    n = rng.randint(1, 8)
    k = rng.randint(0, 6)
    r, c = rng.randrange(n), rng.randrange(n)
    assert knight_exact_ref(n, k, r, c) == oracle(n, k, r, c)
assert best_start_ref(8, 3) in [(r, c) for r in range(8) for c in range(8)]
print("Bài 5 OK; best_start(5, 1) =", best_start_ref(5, 1), "| best_start(8, 4) =", best_start_ref(8, 4))

Bài 5 OK; best_start(5, 1) = (2, 2) | best_start(8, 4) = (3, 3)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP kéo cuốn chiếu (cách nộp):**

```python
class Solution:
    def knightProbability(self, n, k, row, column):
        moves = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
        cur = [[1.0] * n for _ in range(n)]            # h = 0: còn trên bàn -> 1
        for _ in range(k):
            nxt = [[0.0] * n for _ in range(n)]
            for i in range(n):
                for j in range(n):
                    total = 0.0
                    for dr, dc in moves:
                        x, y = i + dr, j + dc
                        if 0 <= x < n and 0 <= y < n:
                            total += cur[x][y]
                    nxt[i][j] = total / 8                # luôn chia 8
            cur = nxt
        return cur[row][column]
```

**Template đệ quy có nhớ:**

```python
@lru_cache(maxsize=None)
def f(h, i, j):
    if not (0 <= i < n and 0 <= j < n): return 0.0
    if h == 0: return 1.0
    return sum(f(h - 1, i + dr, j + dc) for dr, dc in moves) / 8
```

**Câu thần chú:** *`f(h, ô) = trung bình (chia 8) của `f(h-1, ô kề)`; ngoài bàn = 0; `h = 0` trong bàn = 1; luôn chia 8; so số thực bằng sai số.*

**Checklist trước khi submit:**

- [ ] Chia cho `8` (không phải số nước hợp lệ)?
- [ ] Tầng `0` khởi tạo `1.0` cho mọi ô trong bàn; `k = 0` trả `1.0`?
- [ ] Bảng tạo bằng list comprehension (không `[[0] * n] * n`), và `nxt` mới mỗi bước?
- [ ] Kiểm tra biên `0 <= x < n and 0 <= y < n` trước khi truy cập `cur[x][y]`?
- [ ] Không đệ quy không nhớ (`8^k`)?
- [ ] Kiểm thử số thực bằng sai số (không `==`), đối chiếu bằng `Fraction`/đếm số nguyên nếu cần?
- [ ] Đã thử: `n = 1`, `k = 0`, tâm bàn 3x3, góc bàn, `n = 25`, `k = 100`?